In [ ]:
import numpy as np
import geopandas as gpd

from shapely.geometry import LineString
from shapely.ops import snap

from scipy.stats import gaussian_kde
from scipy.interpolate import splprep, splev
from scipy.ndimage import label

from skimage.morphology import skeletonize


gdf = gpd.read_file(
    "../../data/processed/trajectory_clusters.gpkg"
)
shallow = gpd.read_file("../data/processed/shallow_waters.gpkg")
land = gpd.read_file("../../data/processed/land_data.gpkg")

# S57 constraints

shallow_union = unary_union(shallow.geometry)
land_union = unary_union(land.geometry)

In [ ]:

# tolerance buffer
buffered_shallow = shallow_union.buffer(5)

# nearest neighbour ordering

def order_points_nearest_neighbor(points):

    if len(points) <= 2:
        return points

    ordered = [points[0]]

    remaining = list(points[1:])

    while remaining:

        last = ordered[-1]

        dists = np.linalg.norm(
            np.array(remaining) - last,
            axis=1
        )

        idx = np.argmin(dists)

        ordered.append(
            remaining.pop(idx)
        )

    return np.array(ordered)

In [ ]:
centerlines = []
cluster_ids = []
traj_ids = []
depth_readings = []
errors = []


for cluster_id in gdf["cluster"].unique():

    if cluster_id == -1:
        continue

    subset = gdf[
        gdf["cluster"] == cluster_id
    ]

    print(f"\nProcessing cluster {cluster_id}")

    # convert to points

    points = []

    for geom in subset.geometry:

        for x, y in geom.coords:

            points.append([x, y])

    points = np.array(points)

    if len(points) < 5:

        print("Skipped (too few points)")

        continue

    try:

        # kde

        kde = gaussian_kde(points.T, bw_method = 0.05)

        xmin, ymin, xmax, ymax = subset.total_bounds

        padding = 50

        xmin -= padding
        xmax += padding
        ymin -= padding
        ymax += padding

        grid_size = 5

        xgrid = np.arange(
            xmin,
            xmax + grid_size,
            grid_size
        )

        ygrid = np.arange(
            ymin,
            ymax + grid_size,
            grid_size
        )

        if len(xgrid) < 2 or len(ygrid) < 2:

            print("Skipped (grid too small)")

            continue

        X, Y = np.meshgrid(
            xgrid,
            ygrid
        )

        grid_coords = np.vstack([
            X.ravel(),
            Y.ravel()
        ])

        Z = kde(grid_coords).reshape(X.shape)


        threshold = np.percentile(Z, 90)

        binary = Z > threshold

        if not np.any(binary):

            print("empty density)")

            continue

        # skeletonize

        skeleton = skeletonize(binary)

        labeled, num = label(skeleton)

        if num > 1:

            sizes = [
                (labeled == i).sum()
                for i in range(1, num + 1)
            ]

            largest = np.argmax(sizes) + 1

            skeleton = labeled == largest

        coords = np.column_stack(
            np.where(skeleton)
        )

        if len(coords) < 2:

            print("Skipped (invalid skeleton)")

            continue


        ridge_points = np.array([
            (X[i, j], Y[i, j])
            for i, j in coords
        ])

        ridge_points = np.unique(
            ridge_points,
            axis=0
        )

        if len(ridge_points) < 2:

            continue


        ridge_points = order_points_nearest_neighbor(
            ridge_points
        )

        line = LineString(ridge_points)

        #smooth the extracted splice

        if len(ridge_points) >= 4:

            try:

                tck, _ = splprep(
                    [
                        ridge_points[:, 0],
                        ridge_points[:, 1]
                    ],
                    s=10
                )

                u_new = np.linspace(
                    0,
                    1,
                    200
                )

                x_smooth, y_smooth = splev(
                    u_new,
                    tck
                )

                line = LineString(
                    zip(
                        x_smooth,
                        y_smooth
                    )
                )

            except Exception as e:

                print(f"spline extraction failed: {e}")

        # clip to shallow waters

        line = line.intersection(
            buffered_shallow
        )

        if line.is_empty:

            print("Removed after clipping")

            continue

        # check for land intersection

        land_overlap = line.intersection(
            land_union
        )

        overlap_length = land_overlap.length


        if overlap_length > 0 and overlap_length <= 5:

            print("Small land overlap → snapping")

            line = snap(
                line,
                shallow_union.boundary,
                5
            )

        elif overlap_length > 5:

            print("Discarded (large land overlap)")

            continue

        # validate
        if line.is_empty:

            continue

        if not line.is_valid:

            continue

        
        # reconstruction error
        line_coords = np.array(
            line.coords
        )

        dists = []

        for p in points:

            point_dists = np.linalg.norm(
                line_coords - p,
                axis=1
            )

            dists.append(
                np.min(point_dists)
            )

        avg_error = np.mean(dists)

        # store results
        centerlines.append(line)

        cluster_ids.append(cluster_id)

        traj_ids.append(
            subset["traj_id"].unique()
        )

        depth_readings.append(
            subset["DRVAL2"].unique()
        )

        errors.append(avg_error)

        print(f"Cluster {cluster_id} completed")

    except Exception as e:

        print(f"Cluster {cluster_id} failed: {e}")

        continue

In [ ]:
centerline_gdf = gpd.GeoDataFrame({

    "cluster_id": cluster_ids,

    "traj_ids": traj_ids,

    "depths": depth_readings,

    "reconstruction_error": errors,

    "geometry": centerlines

}, geometry="geometry", crs=gdf.crs)

In [ ]:
gdf_centerlines = gpd.GeoDataFrame(
    {"cluster": cluster_ids, "traj_id": traj_ids, "depth_readings": depth_readings},
    geometry=centerlines,
    crs=gdf.crs
)


In [ ]:
gdf_centerlines['string_traj_ids'] = gdf_centerlines['traj_ids'].apply(lambda x: ', '.join(map(str, x)))


In [ ]:
gdf_centerlines.to_file("../../data/processed/corridor_centerlines_skeletionization.gpkg", driver="GPKG")